# Module 02: Core Data Structures


# 2.1 The Doc Object


## 📄 Understanding the `Doc` Object

In spaCy, the `Doc` (Document) is the central data structure. Whenever you process text with the `nlp` object, it returns a `Doc`.

A `Doc` is essentially a sequence of `Token` objects, but it also holds all the linguistic annotations (like sentences, entities, and noun chunks) generated by the pipeline.

Let's explore its core properties!


In [1]:
import spacy

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")

text = "Apple is looking at buying U.K. startup for $1 billion. Tim Cook is the CEO."
doc = nlp(text)

print(f"Document length: {len(doc)} tokens")
print(f"Original text: {doc.text}")


Document length: 18 tokens
Original text: Apple is looking at buying U.K. startup for $1 billion. Tim Cook is the CEO.


## 🔍 Iterating and Indexing

Because a `Doc` is a sequence of tokens, you can interact with it exactly like a Python list.


In [2]:
# Indexing (getting a specific Token)
print(f"First token: '{doc[0]}'")
print(f"Last token: '{doc[-2]}'")

# Iterating
print("\nFirst 5 tokens:")
for token in doc[:5]:
    print(f"- {token.text}")


First token: 'Apple'
Last token: 'CEO'

First 5 tokens:
- Apple
- is
- looking
- at
- buying


## 🏷️ Document-Level Properties

The `Doc` object provides properties that let you access meaningful linguistic units instantly. The most common are:
- `doc.sents`: Sentences
- `doc.ents`: Named Entities
- `doc.noun_chunks`: Base noun phrases


In [3]:
print("=== SENTENCES ===")
for i, sent in enumerate(doc.sents):
    print(f"{i+1}. {sent.text}")

print("\n=== NAMED ENTITIES ===")
for ent in doc.ents:
    print(f"{ent.text:<15} ({ent.label_})")

print("\n=== NOUN CHUNKS ===")
for chunk in doc.noun_chunks:
    print(f"{chunk.text:<15} (Root: {chunk.root.text})")


=== SENTENCES ===
1. Apple is looking at buying U.K. startup for $1 billion.
2. Tim Cook is the CEO.

=== NAMED ENTITIES ===
Apple           (ORG)
U.K.            (GPE)
$1 billion      (MONEY)
Tim Cook        (PERSON)

=== NOUN CHUNKS ===
Apple           (Root: Apple)
U.K.            (Root: U.K.)
Tim Cook        (Root: Cook)
the CEO         (Root: CEO)


## 💾 Serialization (Saving and Loading)

If you process millions of documents, you don't want to re-run the NLP pipeline every time. You can serialize (save) `Doc` objects to a highly compressed binary format using `doc.to_bytes()`.

*Note: To deserialize it later, you must use a `Doc` object created with the exact same vocabulary.*


In [4]:
from spacy.tokens import Doc

# Serialize to bytes
doc_bytes = doc.to_bytes()
print(f"Serialized Doc size: {len(doc_bytes)} bytes")

# Deserialize back to a Doc object
# We initialize an empty Doc with the shared nlp.vocab
loaded_doc = Doc(nlp.vocab).from_bytes(doc_bytes)

print(f"Restored document text: {loaded_doc.text}")


Serialized Doc size: 9769 bytes
Restored document text: Apple is looking at buying U.K. startup for $1 billion. Tim Cook is the CEO.



<br><br>

---

<br><br>


# 2.2 The Token Object


## 🔠 Understanding the `Token` Object

A `Token` represents a single word, punctuation mark, or symbol. 
Tokens carry all the detailed information that spaCy's pipeline has predicted.

We generally divide token attributes into two categories:
1. **Lexical Attributes**: Inherent properties of the word itself (e.g., is it a number? is it uppercase?).
2. **Linguistic Attributes**: Predictions made by the statistical model based on context (e.g., is it a verb? what is its dependency?).


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple earned $1 million in 2023 at spacy.io! ")


## 🔎 Lexical Attributes (Rules-based)

These attributes don't require a statistical model. They are extremely useful for writing fast, rule-based logic (which we will cover in Module 8).


In [2]:
lexical_data = []
for token in doc:
    lexical_data.append({
        "Text": token.text,
        "is_alpha": token.is_alpha,
        "is_punct": token.is_punct,
        "is_space": token.is_space,
        "like_num": token.like_num,   # Understands '1', but also 'one', 'billion'
        "like_url": token.like_url,
        "like_email": token.like_email
    })

pd.DataFrame(lexical_data)


,Text,is_alpha,is_punct,is_space,like_num,like_url,like_email
0,Apple,True,False,False,False,False,False
1,earned,True,False,False,False,False,False
2,$,False,False,False,False,False,False
3,1,False,False,False,True,False,False
4,million,True,False,False,True,False,False
5,in,True,False,False,False,False,False
6,2023,False,False,False,True,False,False
7,at,True,False,False,False,False,False
8,spacy.io,False,False,False,False,True,False
9,!,False,True,False,False,False,False


## 🧠 Linguistic Attributes (Model Predictions)

These rely on the statistical pipeline. They change depending on the context of the sentence.


In [3]:
linguistic_data = []
for token in doc:
    # Skip spaces to keep the table clean
    if token.is_space: 
        continue
        
    linguistic_data.append({
        "Text": token.text,
        "Lemma (Base Form)": token.lemma_,
        "POS (Simple)": token.pos_,
        "Tag (Detailed)": token.tag_,
        "Dependency": token.dep_
    })

pd.DataFrame(linguistic_data)


,Text,Lemma (Base Form),POS (Simple),Tag (Detailed),Dependency
0,Apple,Apple,PROPN,NNP,nsubj
1,earned,earn,VERB,VBD,ROOT
2,$,$,SYM,$,quantmod
3,1,1,NUM,CD,compound
4,million,million,NUM,CD,dobj
5,in,in,ADP,IN,prep
6,2023,2023,NUM,CD,pobj
7,at,at,ADP,IN,prep
8,spacy.io,spacy.io,NUM,CD,pobj
9,!,!,PUNCT,.,punct


## 🌳 Token Navigation (The Dependency Tree)

Tokens are connected to each other via a syntactic dependency tree. You can navigate this tree using token properties:
- `token.head`: The parent of the token.
- `token.children`: The tokens that depend on this token.
- `token.ancestors`: All parents up to the root of the sentence.


In [4]:
sentence = nlp("The clever dog quickly chased the red ball.")

# Let's look at the word 'chased' (the root of the sentence)
verb_token = sentence[4]
print(f"Target Token: {verb_token.text}")

# Who is the head of 'chased'?
print(f"Head: {verb_token.head.text}")

# What are the children of 'chased'?
children = [child.text for child in verb_token.children]
print(f"Children: {children}")

print("\nLet's look at the word 'dog':")
dog_token = sentence[2]
print(f"Ancestors of 'dog': {[ancestor.text for ancestor in dog_token.ancestors]}")


Target Token: chased
Head: chased
Children: ['dog', 'quickly', 'ball', '.']

Let's look at the word 'dog':
Ancestors of 'dog': ['chased']



<br><br>

---

<br><br>


# 2.3 The Span Object


## 📏 Understanding the `Span` Object

A `Span` is a continuous sequence of tokens from a `Doc`.

While a `Token` is a single word, and a `Doc` is the whole text, a `Span` is any slice in between. Sentences (`doc.sents`) and Named Entities (`doc.ents`) are both actually returned as `Span` objects!


In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("I traveled to New York City last weekend.")

# Creating a span by slicing the Doc
# New York City is at indices 3, 4, 5
city_span = doc[3:6]

print(f"Span text: '{city_span.text}'")
print(f"Type: {type(city_span)}")


Span text: 'New York City'
Type: <class 'spacy.tokens.span.Span'>


## 🏷️ Span Properties

Because spans are derived from a `Doc`, they inherit a lot of powerful context. A span knows exactly where it belongs in the original text.


In [2]:
print(f"Span start index: {city_span.start}")
print(f"Span end index: {city_span.end}")

# A span can figure out its root token (the central word that connects it to the sentence)
print(f"Span root token: '{city_span.root.text}'")

# A span knows which sentence it belongs to
print(f"Belongs to sentence: '{city_span.sent.text}'")


Span start index: 3
Span end index: 6
Span root token: 'City'
Belongs to sentence: 'I traveled to New York City last weekend.'


## 🎨 Labeled Spans

Often, you want to create a custom Span and assign a label to it (for example, if you are building your own Named Entity Recognizer). You can manually create `Span` objects and add them to `doc.ents`.


In [3]:
from spacy.tokens import Span

doc2 = nlp("David Bowies was a musical genius.")

# Let's say our model failed to recognize "David Bowies" as a PERSON.
# We can manually create a Span with the label "PERSON"

# Span(doc, start_idx, end_idx, label)
new_ent = Span(doc2, 0, 2, label="PERSON")
print(f"Created manual entity: {new_ent.text} ({new_ent.label_})")

# We can add this to the document's entities
doc2.set_ents([new_ent], default="unmodified")

print("\nDocument Entities:")
for ent in doc2.ents:
    print(f"- {ent.text} [{ent.label_}]")


Created manual entity: David Bowies (PERSON)

Document Entities:
- David Bowies [PERSON]


## 📂 Span Groups

Sometimes, spans overlap. For example, in "New York City", you might want an entity for "New York City" (Location) and "New York" (State).

`doc.ents` strictly **does not allow overlapping spans**.

To solve this, spaCy introduced **Span Groups** (`doc.spans`). It's a dictionary where you can store lists of spans, overlapping or not!


In [4]:
doc3 = nlp("I live in New York City.")

span_city = doc3[3:6]  # New York City
span_state = doc3[3:5] # New York

# We save them in a custom span group named 'locations'
doc3.spans["locations"] = [span_city, span_state]

print("Custom Span Group 'locations':")
for span in doc3.spans["locations"]:
    print(f"- {span.text}")


Custom Span Group 'locations':
- New York City
- New York



<br><br>

---

<br><br>


# 2.4 Vocab, StringStore, and Lexemes


## 💾 Deep Dive into Memory Efficiency

In Module 1, we briefly touched on spaCy's memory management. Let's look closer.

NLP involves handling massive arrays of text. If you store the string "the" a million times, you waste memory. spaCy stores strings exactly once in the `StringStore`, converting them to 64-bit integer hashes.


In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Adding and retrieving from the StringStore
# You can add a new word to the vocabulary
hash_id = nlp.vocab.strings.add("spacy_rocks")
print(f"Hash ID for 'spacy_rocks': {hash_id}")

# You can look it up by string or by hash
print(f"Lookup by string: {nlp.vocab.strings['spacy_rocks']}")
print(f"Lookup by hash: {nlp.vocab.strings[hash_id]}")


Hash ID for 'spacy_rocks': 11057445964824623400
Lookup by string: 11057445964824623400
Lookup by hash: spacy_rocks


## 🧩 Lexemes

Whenever you access the `nlp.vocab` using a string or a hash, spaCy returns a **Lexeme**.

A `Token` is a word *in a specific context* (in a sentence).
A `Lexeme` is a word *in the dictionary* (out of context).

Lexemes don't have POS tags or dependencies (because those depend on the sentence context), but they do have lexical attributes like `is_alpha` or `shape_`.


In [2]:
lexeme = nlp.vocab["Batman"]

print(f"Lexeme text: {lexeme.text}")
print(f"Lexeme hash: {lexeme.orth}")
print(f"Is title case?: {lexeme.is_title}")
print(f"Shape: {lexeme.shape_}")

# Notice this will throw an error if you uncomment it, because a Lexeme has no POS tag!
# print(lexeme.pos_)


Lexeme text: Batman
Lexeme hash: 15780808606389958345
Is title case?: True
Shape: Xxxxx


## ⚠️ The Danger of Unshared Vocabularies

Because of this hash system, two different `nlp` objects do not share the same `StringStore` unless explicitly designed to. 

If you create a `Doc` with `nlp_english` and try to add a Span to it from `nlp_german`, spaCy will crash or give garbage data, because Hash `12345` means "coffee" in English but might mean something completely different (or not exist) in the German vocabulary!

**Golden Rule:** Always ensure your `Doc`, `Token`, and `Span` objects share the exact same `Vocab` object when interacting.



<br><br>

---

<br><br>


# 2.5 Advanced Data Structures


## 🛠️ Manual Doc Creation

Sometimes you already have pre-tokenized text (e.g., from another system) and you want to bring it into spaCy without running it through the spaCy tokenizer.

You can manually instantiate a `Doc` object by passing in the vocabulary and a list of words.


In [1]:
import spacy
from spacy.tokens import Doc

nlp = spacy.load("en_core_web_sm")

words = ["Hello", "world", "!"]
spaces = [True, False, False]  # Whether the word is followed by a space

# Manually create the Doc
manual_doc = Doc(nlp.vocab, words=words, spaces=spaces)

print(f"Constructed Text: '{manual_doc.text}'")
print(f"Number of tokens: {len(manual_doc)}")


Constructed Text: 'Hello world!'
Number of tokens: 3


## ✂️ Merging Tokens (Retokenization)

Often, multiple tokens represent a single logical unit. For example, "Los Angeles" is two tokens, but conceptually it's one city. You might want to merge them into a single token so your downstream code is simpler.

spaCy provides a `doc.retokenize()` context manager for this.


In [2]:
doc = nlp("I flew to Los Angeles yesterday.")

print("Before merge:", [t.text for t in doc])

# We know Los Angeles is at index 3 and 4
span_to_merge = doc[3:5]

# Use the retokenizer context manager
with doc.retokenize() as retokenizer:
    # We merge the span into a single token
    retokenizer.merge(span_to_merge)

print("After merge:", [t.text for t in doc])
print(f"Now token 3 is: '{doc[3].text}'")


Before merge: ['I', 'flew', 'to', 'Los', 'Angeles', 'yesterday', '.']
After merge: ['I', 'flew', 'to', 'Los Angeles', 'yesterday', '.']
Now token 3 is: 'Los Angeles'


## 📋 Best Practices for Retokenization

When you merge tokens, what happens to their properties? 
By default, the new merged token inherits the properties (like POS tag, lemma, and dependency) of the **root** token of the span.

However, you can explicitly set the properties of the newly merged token:


In [3]:
doc2 = nlp("The CEO of Apple Inc. announced a new product.")

with doc2.retokenize() as retokenizer:
    # We want to merge 'Apple' and 'Inc.'
    attrs = {
        "LEMMA": "Apple Incorporated", 
        "POS": "PROPN"
    }
    retokenizer.merge(doc2[3:5], attrs=attrs)

print("Merged text:", doc2[3].text)
print("Merged lemma:", doc2[3].lemma_)
print("Merged POS:", doc2[3].pos_)


Merged text: Apple Inc.
Merged lemma: Apple Incorporated
Merged POS: PROPN


## 🎉 Summary of Module 2

You now have total control over spaCy's core data structures: `Doc`, `Token`, `Span`, and `Vocab`.

You know how to create them, navigate them, slice them, manually label them, and even mutate the document by merging tokens together!

In **Module 3**, we will look at **The NLP Pipeline**. You will learn how the components that assign tags and entities actually work, and how to add, remove, and debug pipeline steps.
